# 인사 데이터 텍스트 스플릿 및 청킹

### 라이브러리 설치

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 라이브러리 선언

In [2]:
# JSON 파일 읽기/쓰기 라이브러리
import json
# 파일 경로 처리 라이브러리
from pathlib import Path
# 운영체제 환경변수 읽기
import os
# .env 파일의 환경변수를 불러오는 라이브러리
from dotenv import load_dotenv
# 텍스트 청크 분할 라이브러리
from langchain_text_splitters import RecursiveCharacterTextSplitter
# LangChain 문서 객체 (텍스트 + 메타데이터를 함께 관리)
from langchain_core.documents import Document

print('라이브러리 로딩 완료!')

라이브러리 로딩 완료!


### 환경 설정 *** 경로 확인 필요

In [ ]:
# .env 파일의 설정값을 환경변수로 불러옴
load_dotenv()

# 변환된 JSONL 파일이 있는 폴더 경로
INPUT_DIR  = Path(os.getenv('INPUT_DIR',  '../JSONL 변환/output'))
# 처리 결과를 저장할 폴더 경로
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

# 출력 폴더가 없으면 자동으로 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 청킹 하이퍼파라미터 설정
CHUNK_SIZE    = int(os.getenv('CHUNK_SIZE',    '200'))  # 청크 크기 (글자 수 기준)
CHUNK_OVERLAP = int(os.getenv('CHUNK_OVERLAP', '50'))   # 앞뒤 청크 간 겹치는 글자 수

print(f'입력 디렉토리: {INPUT_DIR}')
print(f'출력 디렉토리: {OUTPUT_DIR}')
print('\n설정된 하이퍼파라미터:')
print(f'  청크 크기   : {CHUNK_SIZE}')
print(f'  오버랩      : {CHUNK_OVERLAP}')
print('\n설정값 영향:')
print('  - 큰 청크: 문맥 유지↑, 검색 정밀도↓')
print('  - 많은 오버랩: 문맥 단절 방지, 중복↑')

# 1. 데이터 로딩

### 1-1. JSONL 파일 로드

In [4]:
# 입력 폴더에서 .jsonl 파일 목록을 이름순으로 가져옴
jsonl_files = sorted(INPUT_DIR.glob('*.jsonl'))

if not jsonl_files:
    print(f'JSONL 파일 없음: {INPUT_DIR}')
    print('JSONL 변환 노트북을 먼저 실행해 주세요.')
else:
    # 파일 이름을 키로, Document 리스트를 값으로 저장하는 딕셔너리
    # Document: LangChain의 문서 객체 (page_content + metadata)
    doc_sets = {}
    # 첫 번째 파일의 샘플을 출력하기 위한 변수
    first_sample = None

    for path in jsonl_files:
        docs = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line.strip())

                # JSONL 레코드를 Document 객체로 변환
                # page_content: 분할 대상 텍스트
                # metadata    : 분할 후에도 각 청크에 유지할 정보
                doc = Document(
                    page_content=record['embedding_text'],
                    metadata={
                        'employee_id':      record['employee_id'],
                        'department':       record['department'],
                        'position':         record['position'],
                        'embedding_vector': record['embedding_vector'],
                        'source':           path.name,
                    }
                )
                docs.append(doc)

        doc_sets[path.stem] = docs
        print(f'  로딩: {path.name}  ({len(docs):,}건)')

        # 첫 번째 파일의 첫 번째 Document를 샘플로 저장
        if first_sample is None:
            first_sample = docs[0]

    print(f'\n로딩 완료! 총 {len(doc_sets)}개 파일')
    print('\n첫 번째 Document 샘플:')
    print('-' * 50)
    print(f'page_content: {first_sample.page_content}')
    print(f'metadata    : {first_sample.metadata}')
    print('-' * 50)

  로딩: 급여정보_정제.jsonl  (2,000건)
  로딩: 기본인사정보_정제.jsonl  (2,000건)
  로딩: 역량성과_정제.jsonl  (2,000건)
  로딩: 통합인사정보_정제.jsonl  (2,000건)

로딩 완료! 총 4개 파일

첫 번째 Document 샘플:
--------------------------------------------------
page_content: 지준서 마케팅부 사원 32000000 우리은행
metadata    : {'employee_id': 'EMP0001', 'department': '마케팅부', 'position': '사원', 'embedding_vector': [], 'source': '급여정보_정제.jsonl'}
--------------------------------------------------


# 2. 텍스트 정규화

### 2-1. 공백 정규화

In [5]:
print('텍스트 정규화 시작...\n')

def clean(text):
    # 앞뒤 공백 제거
    text = text.strip()
    words = text.split()
    text = ' '.join(words)
    return text


normalize_count = 0

for file_name, docs in doc_sets.items():
    for doc in docs:
        original   = doc.page_content
        normalized = clean(original)
        if original != normalized:
            normalize_count += 1
        # Document의 page_content를 정규화된 텍스트로 교체
        doc.page_content = normalized

print(f'정규화 완료!')
print(f'  정규화 적용된 레코드 수: {normalize_count:,}건')
print('\n정규화 후 샘플:')
print('-' * 50)
print(first_sample.page_content)
print('-' * 50)

텍스트 정규화 시작...

정규화 완료!
  정규화 적용된 레코드 수: 0건

정규화 후 샘플:
--------------------------------------------------
지준서 마케팅부 사원 32000000 우리은행
--------------------------------------------------


# 3. 스플릿 및 청킹

### 3-1. 텍스트 분할기 설정

In [6]:
# RecursiveCharacterTextSplitter: 구분자 우선순위에 따라 자연스럽게 분할
# chunk_size보다 짧은 텍스트는 분할하지 않고 그대로 유지
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', '!', '?', ',', ' ', '']
)

print('분할기 설정 완료!')

분할기 설정 완료!


### 3-2. 청킹 실행

In [7]:
print('텍스트 청크 분할 시작...\n')

# 파일별 청킹 결과를 저장하는 딕셔너리
chunked_sets = {}

for file_name, docs in doc_sets.items():
    # split_documents: Document 리스트를 받아 청크로 분할
    # 메타데이터(employee_id, department 등)는 자동으로 각 청크에 복사됨
    chunks = text_splitter.split_documents(docs)
    chunked_sets[file_name] = chunks

    print(f'청크 분할 결과: [{file_name}]')
    print(f'  원본 레코드 수: {len(docs):,}건')
    print(f'  생성된 청크 수: {len(chunks):,}건')
    print('\n  첫 번째 청크 샘플:')
    print('  ' + '-' * 48)
    print(f'  {chunks[0].page_content}')
    print('  ' + '-' * 48)
    print(f'  청크 메타데이터: {chunks[0].metadata}\n')

print('청킹 완료!')

텍스트 청크 분할 시작...

청크 분할 결과: [급여정보_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 2,000건

  첫 번째 청크 샘플:
  ------------------------------------------------
  지준서 마케팅부 사원 32000000 우리은행
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'department': '마케팅부', 'position': '사원', 'embedding_vector': [], 'source': '급여정보_정제.jsonl'}

청크 분할 결과: [기본인사정보_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 2,000건

  첫 번째 청크 샘플:
  ------------------------------------------------
  지준서 마케팅부 사원 브랜드팀 팀원 2024-05-02 jijunseo16@nate.com
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'department': '마케팅부', 'position': '사원', 'embedding_vector': [], 'source': '기본인사정보_정제.jsonl'}

청크 분할 결과: [역량성과_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 2,000건

  첫 번째 청크 샘플:
  ------------------------------------------------
  지준서 마케팅부 사원 62
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'department': '마케팅부', 'position': '사원', 'embedding_vect

# 4. 검증

### 4-1. 데이터 소실 여부 확인

In [8]:
print('데이터 소실 여부 확인 중...')
print('-' * 50)

for file_name in doc_sets:
    original_count = len(doc_sets[file_name])
    chunk_count    = len(chunked_sets[file_name])

    # 청킹 후 건수는 같거나 많아야 함 (분할되면 늘어남)
    if chunk_count >= original_count:
        status = '정상'
    else:
        status = '소실 의심'

    print(f'  {file_name}')
    print(f'  원본 {original_count:,}건 → 청크 {chunk_count:,}건  [{status}]\n')

print('-' * 50)
print('데이터 소실 여부 확인 완료!')

데이터 소실 여부 확인 중...
--------------------------------------------------
  급여정보_정제
  원본 2,000건 → 청크 2,000건  [정상]

  기본인사정보_정제
  원본 2,000건 → 청크 2,000건  [정상]

  역량성과_정제
  원본 2,000건 → 청크 2,000건  [정상]

  통합인사정보_정제
  원본 2,000건 → 청크 2,000건  [정상]

--------------------------------------------------
데이터 소실 여부 확인 완료!


### 4-2. 메타데이터 유지 확인

In [9]:
print('메타데이터 유지 확인 중...')
print('-' * 50)

# 데이터구조정의서 기준 필수 메타데이터
required_meta = ['employee_id', 'department', 'position', 'embedding_vector']

for file_name, chunks in chunked_sets.items():
    missing_count = 0

    for chunk in chunks:
        for field in required_meta:
            if field not in chunk.metadata:
                missing_count += 1
                break

    status = '정상' if missing_count == 0 else f'누락 {missing_count}건'
    print(f'  {file_name}: [{status}]')

print('-' * 50)
print('메타데이터 유지 확인 완료!')

메타데이터 유지 확인 중...
--------------------------------------------------
  급여정보_정제: [정상]
  기본인사정보_정제: [정상]
  역량성과_정제: [정상]
  통합인사정보_정제: [정상]
--------------------------------------------------
메타데이터 유지 확인 완료!


# 5. 결과 저장

In [10]:
print('결과 저장 중...\n')

for file_name, chunks in chunked_sets.items():
    out_path = OUTPUT_DIR / f'{file_name}.jsonl'

    with open(out_path, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            # Document 객체를 다시 JSONL 레코드 형태로 변환
            record = {
                'employee_id':      chunk.metadata['employee_id'],
                'department':       chunk.metadata['department'],
                'position':         chunk.metadata['position'],
                'embedding_text':   chunk.page_content,
                'embedding_vector': chunk.metadata['embedding_vector'],
            }
            # ensure_ascii=False: 한글을 그대로 저장
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f'  저장: {out_path.name}  ({len(chunks):,}건)')

print('\n모든 파일 저장 완료!')

결과 저장 중...

  저장: 급여정보_정제.jsonl  (2,000건)
  저장: 기본인사정보_정제.jsonl  (2,000건)
  저장: 역량성과_정제.jsonl  (2,000건)
  저장: 통합인사정보_정제.jsonl  (2,000건)

모든 파일 저장 완료!
